In [6]:
"""
Metrics runner for your labeled sample.

Reads your *autofilled & manually-reviewed* sheet from:
  C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\Stratified Sample
and computes:

A) Quantified prevalence (population-level, with stratified weights & CIs)
   - yaml_signal_true, build_true, at_true, ci_runs_true
   - 2×2 truth cohorts (CI-only, Build-only, Both, None)

B) Error rates / quality metrics (precision, recall) for each detector
   - YAML_pred vs yaml_signal_true (and optionally vs ci_runs_true)
   - Build_pred vs build_true
   - AT_pred vs at_true
   Each: overall, by provider, by group_pred (CI-only / Build-only / Both / None)
   Uses stratified expansion weights (N_h / n_h) from cohort_table.csv

Also exports:
  - Weighted confusion matrices (overall & by group_pred)
  - Failure modes (reason_codes) by group_pred (weighted shares)

Outputs are saved in the same folder.

Run (PowerShell):
  & "C:\Python312\python.exe" "C:\path\to\metrics_runner.py"
"""

import os
import re
import math
import pandas as pd
import numpy as np
from pathlib import Path

# -------------------- CONFIG --------------------
BASE_DIR = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\Stratified Sample"
OUTPUT_DIR = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\Stratified Sample\Metrics"

# Try these filenames (the script will pick the first that exists)
MANUAL_FILES = [
    "manual_review_sheet_autofilled.csv",
    "Manual_Review_Sheet__Prefilled_RQ1.csv",
    "manual_review_sheet_prefilled_v2.csv",
    "manual_review_sheet_prefilled.csv"
]
COHORT_FILE = "cohort_table.csv"   # from your sampling script

OUT_DIR = BASE_DIR  # save alongside your sheet

CONF_LEVEL = 0.90
Z = 1.645  # z-score for 90%

# -------------------- IO HELPERS --------------------
def first_existing(base, names):
    for n in names:
        p = Path(base) / n
        if p.exists():
            return str(p)
    raise FileNotFoundError(f"Could not find any of: {names} in {base}")

def to_int01(x):
    if pd.isna(x): return None
    if isinstance(x, (int, np.integer)): return int(x != 0)
    if isinstance(x, float): 
        if np.isnan(x): return None
        return int(x != 0.0)
    s = str(x).strip().lower()
    if s in {"1","true","t","yes","y"}: return 1
    if s in {"0","false","f","no","n"}: return 0
    return None

# Wilson interval for a binomial proportion (k successes in n trials)
def wilson_ci(k, n, z=Z):
    if n is None or n <= 0:
        return (None, None)
    p = k / n
    denom = 1 + z**2 / n
    center = (p + z**2 / (2*n)) / denom
    half = (z / denom) * math.sqrt((p*(1-p)/n) + (z**2 / (4*n**2)))
    lo = max(0.0, center - half)
    hi = min(1.0, center + half)
    return (lo, hi)

# -------------------- LOAD --------------------
manual_path = first_existing(BASE_DIR, MANUAL_FILES)
cohort_path = str(Path(BASE_DIR) / COHORT_FILE)

df = pd.read_csv(manual_path)
cohort = pd.read_csv(cohort_path)

# Ensure required columns exist
needed_cols = [
    "html_url","full_name","provider","stratum",
    "YAML_pred","Build_pred","AT_pred","group_pred",
    "yaml_signal_true","build_true","at_true","ci_runs_true","group_true",
    "reason_codes"
]
for c in needed_cols:
    if c not in df.columns:
        df[c] = np.nan

# Normalize 0/1
for col in ["YAML_pred","Build_pred","AT_pred","yaml_signal_true","build_true","at_true","ci_runs_true"]:
    df[col] = df[col].apply(to_int01)

# Ensure group_pred exists (derive from preds if missing)
def derive_group(y, b):
    if y==1 and b==0: return "CI-only"
    if y==0 and b==1: return "Build-only"
    if y==1 and b==1: return "Both"
    if y==0 and b==0: return "None"
    return "Unknown"

if "group_pred" not in df.columns or df["group_pred"].isna().all():
    df["group_pred"] = df.apply(lambda r: derive_group(r["YAML_pred"], r["Build_pred"]), axis=1)

# Derive group_true if missing
def derive_group_true(y_true, b_true):
    y = to_int01(y_true); b = to_int01(b_true)
    if y is None or b is None: return np.nan
    return derive_group(y, b)

if "group_true" not in df.columns or df["group_true"].isna().all():
    df["group_true"] = df.apply(lambda r: derive_group_true(r["yaml_signal_true"], r["build_true"]), axis=1)

# -------------------- STRATUM WEIGHTS --------------------
# cohort_table has: YAML_pred, Build_pred, AT_pred, stratum, N, pct
N_total = int(cohort["N"].sum())
Nh_map = dict(zip(cohort["stratum"], cohort["N"]))  # stratum -> N_h

# Count labeled per stratum (n_h) per truth target (we'll compute per metric)
def stratum_counts(df_in, truth_col):
    # only rows with non-null truth count toward n_h
    tmp = df_in[["stratum", truth_col]].copy()
    tmp = tmp[tmp[truth_col].apply(lambda x: x in (0,1))]
    nh = tmp.groupby("stratum").size().rename("n_h")
    return nh.to_dict()

# Per-row expansion weight w_i = N_h / n_h(stratum of row)  (computed per truth target)
def compute_weights(df_in, truth_col):
    nh = stratum_counts(df_in, truth_col)
    w = []
    for _, r in df_in.iterrows():
        s = r["stratum"]
        N_h = Nh_map.get(s, None)
        n_h = nh.get(s, None)
        if (N_h is None) or (n_h is None) or n_h == 0 or pd.isna(r[truth_col]):
            w.append(np.nan)
        else:
            w.append(N_h / n_h)
    return np.array(w, dtype="float64")

# -------------------- PREVALENCE (STRATIFIED, WITH FPC) --------------------
def stratified_prevalence(df_in, truth_col):
    # Compute per-stratum p_h and n_h from labeled rows
    labeled = df_in[df_in[truth_col].apply(lambda x: x in (0,1))].copy()
    if labeled.empty:
        return dict(est=None, lo=None, hi=None, n_total=0, labeled=0)

    g = labeled.groupby("stratum")
    ph = g[truth_col].mean()  # sample proportion per stratum
    nh = g.size()
    res = []

    # Combine with population N_h
    var_sum = 0.0
    est = 0.0
    for s, p_h in ph.items():
        N_h = Nh_map.get(s, 0)
        n_h = nh[s]
        W_h = N_h / N_total if N_total > 0 else 0.0
        est += W_h * p_h
        # Stratified variance approx with FPC for proportions under SRSWOR within stratum:
        # Var(p̂_h) ≈ p_h(1-p_h)/(n_h) * (1 - n_h/N_h)
        if n_h > 0 and N_h > 1:
            var_h = (p_h * (1 - p_h) / n_h) * (1 - n_h / N_h)
            var_sum += (W_h ** 2) * var_h

    se = math.sqrt(max(var_sum, 0.0))
    lo = max(0.0, est - Z * se)
    hi = min(1.0, est + Z * se)
    return dict(est=est, lo=lo, hi=hi, n_total=int(nh.sum()), labeled=int(len(labeled)))

def prevalence_table(df_in):
    rows = []
    for col in ["yaml_signal_true","build_true","at_true","ci_runs_true"]:
        res = stratified_prevalence(df_in, col)
        rows.append({
            "measure": col,
            "estimate": res["est"],
            "ci90_lo": res["lo"],
            "ci90_hi": res["hi"],
            "labeled_rows": res["labeled"],
            "sum_nh_across_strata": res["n_total"]
        })
    # Truth cohorts (group_true)
    # Compute weighted shares over 4 groups
    labeled = df_in[~df_in["group_true"].isna()].copy()
    if not labeled.empty:
        # expansion weights built from yaml truth availability is tricky; rebuild using both yaml & build truths present
        mask = df_in["yaml_signal_true"].apply(lambda x: x in (0,1)) & df_in["build_true"].apply(lambda x: x in (0,1))
        df_g = df_in[mask].copy()
        w = compute_weights(df_g, "yaml_signal_true")  # nh is same mask as build_true mask
        df_g["w"] = w
        grp = (df_g
               .groupby("group_true", dropna=False)["w"]
               .sum(min_count=1))
        total_w = grp.sum()
        for gname, wsum in grp.items():
            if total_w and pd.notna(wsum):
                rows.append({
                    "measure": "group_true_share",
                    "group": gname,
                    "estimate": float(wsum / total_w),
                    "ci90_lo": None,
                    "ci90_hi": None,
                    "labeled_rows": int(mask.sum()),
                    "sum_nh_across_strata": int(mask.sum())
                })
    return pd.DataFrame(rows)

# -------------------- DETECTOR METRICS (PRECISION/RECALL) --------------------
def weighted_confusion(df_in, pred_col, truth_col):
    # Only rows with truth in {0,1} and pred in {0,1}
    d = df_in[df_in[truth_col].apply(lambda x: x in (0,1)) & df_in[pred_col].apply(lambda x: x in (0,1))].copy()
    if d.empty:
        return dict(TP=0, FP=0, TN=0, FN=0, support=0)
    # expansion weights
    w = compute_weights(d, truth_col)
    d["w"] = w
    # weighted counts
    TP = float(d[(d[pred_col]==1) & (d[truth_col]==1)]["w"].sum())
    FP = float(d[(d[pred_col]==1) & (d[truth_col]==0)]["w"].sum())
    TN = float(d[(d[pred_col]==0) & (d[truth_col]==0)]["w"].sum())
    FN = float(d[(d[pred_col]==0) & (d[truth_col]==1)]["w"].sum())
    return dict(TP=TP, FP=FP, TN=TN, FN=FN, support=float(d["w"].sum()))

def pr_from_conf(C):
    TP, FP, FN = C["TP"], C["FP"], C["FN"]
    precision = TP / (TP + FP) if (TP + FP) > 0 else None
    recall    = TP / (TP + FN) if (TP + FN) > 0 else None
    # Wilson CIs treating weighted counts as k,n (approx)
    p_lo, p_hi = (None, None) if precision is None else wilson_ci(TP, TP+FP, z=Z)
    r_lo, r_hi = (None, None) if recall    is None else wilson_ci(TP, TP+FN, z=Z)
    return dict(precision=precision, recall=recall,
                precision_ci90_lo=p_lo, precision_ci90_hi=p_hi,
                recall_ci90_lo=r_lo, recall_ci90_hi=r_hi)

def metrics_block(df_in, scope_label):
    rows = []
    targets = [
        ("YAML_pred",  "yaml_signal_true", "YAML vs YAML truth"),
        ("YAML_pred",  "ci_runs_true",     "YAML vs CI actually runs"),  # optional interpretation
        ("Build_pred", "build_true",       "Build vs Build truth"),
        ("AT_pred",    "at_true",          "AT vs AT truth"),
    ]
    conf_rows = []
    for pred, truth, desc in targets:
        C = weighted_confusion(df_in, pred, truth)
        PR = pr_from_conf(C)
        row = dict(scope=scope_label, pred=pred, truth=truth, desc=desc)
        row.update(PR)
        row.update(C)
        rows.append(row)
        conf = dict(scope=scope_label, pred=pred, truth=truth, desc=desc)
        conf.update(C)
        conf_rows.append(conf)
    return pd.DataFrame(rows), pd.DataFrame(conf_rows)

# -------------------- BY GROUP / PROVIDER --------------------
def by_category(df_in, cat_col, name):
    all_metrics = []
    all_conf = []
    for cat, g in df_in.groupby(cat_col, dropna=False):
        label = f"{name}:{cat}"
        m, c = metrics_block(g, label)
        all_metrics.append(m); all_conf.append(c)
    if all_metrics:
        return pd.concat(all_metrics, ignore_index=True), pd.concat(all_conf, ignore_index=True)
    else:
        return pd.DataFrame(), pd.DataFrame()

# -------------------- FAILURE MODES --------------------
def failure_modes_by_group(df_in):
    # Expansion weights from yaml truth (most relevant to CI cohort issues)
    df_tmp = df_in.copy()
    df_tmp["w"] = compute_weights(df_tmp, "yaml_signal_true")
    # explode reason_codes
    rc = df_tmp[["group_pred","reason_codes","w"]].copy()
    rc["reason_codes"] = rc["reason_codes"].fillna("").astype(str)
    rc = rc[rc["reason_codes"].str.strip() != ""]
    if rc.empty:
        return pd.DataFrame()
    rc["reason_codes"] = rc["reason_codes"].str.split(",")
    rc = rc.explode("reason_codes")
    rc["reason_codes"] = rc["reason_codes"].str.strip()
    # aggregate weighted counts
    grp = rc.groupby(["group_pred","reason_codes"]).agg(weighted_count=("w","sum")).reset_index()
    # within-group shares
    totals = grp.groupby("group_pred")["weighted_count"].sum().rename("group_total")
    grp = grp.merge(totals, on="group_pred", how="left")
    grp["share_in_group"] = np.where(grp["group_total"]>0, grp["weighted_count"] / grp["group_total"], np.nan)
    return grp.sort_values(["group_pred","weighted_count"], ascending=[True, False])

# -------------------- RUN --------------------
# Overall metrics
metrics_overall, conf_overall = metrics_block(df, "overall")

# By provider (multi-label providers are strings like 'github_actions;gitlab' -> split into multiple rows)
df_prov = df.copy()
df_prov["provider"] = df_prov["provider"].fillna("unknown").astype(str)
df_prov = df_prov.assign(provider=df_prov["provider"].str.split(";"))
df_prov = df_prov.explode("provider").assign(provider=lambda s: s["provider"].str.strip().replace("", "unknown"))
metrics_by_provider, conf_by_provider = by_category(df_prov, "provider", "provider")

# By predicted group (CI-only / Build-only / Both / None)
metrics_by_group, conf_by_group = by_category(df, "group_pred", "group_pred")

# Prevalence table & cohort shares
prev_tbl = prevalence_table(df)

# Failure modes
fail_modes = failure_modes_by_group(df)

# Coverage (how many labeled)
coverage_rows = []
for col in ["yaml_signal_true","build_true","at_true","ci_runs_true"]:
    n_lab = df[col].apply(lambda x: x in (0,1)).sum()
    coverage_rows.append({"field": col, "labeled_rows": int(n_lab), "total_sample_rows": int(len(df))})
coverage = pd.DataFrame(coverage_rows)

# -------------------- SAVE --------------------
OUT_DIR = Path(OUTPUT_DIR)
OUT_DIR.mkdir(parents=True, exist_ok=True)

def save_csv(df_in, fname):
    path = OUT_DIR / fname
    df_in.to_csv(path, index=False, encoding="utf-8-sig")
    print(f"Saved -> {path}")

save_csv(metrics_overall,     "metrics_overall_90CI_weighted.csv")
save_csv(conf_overall,        "confusion_matrices_overall_weighted.csv")
save_csv(metrics_by_provider, "metrics_by_provider_90CI_weighted.csv")
save_csv(conf_by_provider,    "confusion_matrices_by_provider_weighted.csv")
save_csv(metrics_by_group,    "metrics_by_group_pred_90CI_weighted.csv")
save_csv(conf_by_group,       "confusion_matrices_by_group_pred_weighted.csv")
save_csv(prev_tbl,            "prevalence_estimates_stratified_90CI.csv")
save_csv(fail_modes,          "failure_modes_by_group_pred_weighted.csv")
save_csv(coverage,            "labeling_coverage_report.csv")

print("\nDone. Key files written in your 'Metrics' subfolder.")

Saved -> C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\Stratified Sample\Metrics\metrics_overall_90CI_weighted.csv
Saved -> C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\Stratified Sample\Metrics\confusion_matrices_overall_weighted.csv
Saved -> C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\Stratified Sample\Metrics\metrics_by_provider_90CI_weighted.csv
Saved -> C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\Stratified Sample\Metrics\confusion_matrices_by_provider_weighted.csv
Saved -> C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\Stratified Sample\Metrics\metrics_by_group_pred_90CI_weighted.csv
Saved -> C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\Stratified Sample\Metrics\confusion_matrices_by_group_pred_weighted.csv
Saved -> C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\Stratified Sample\Metrics\prevalence_estimates_stratified_90CI.csv
Saved -> C:\Andro

C:\Users\gilla\AppData\Local\Temp\ipykernel_6852\3834581991.py:280: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(all_metrics, ignore_index=True), pd.concat(all_conf, ignore_index=True)
